In [1]:
# =========================
# Step 2: Extract feature importance
# - Text-only + L1 Logistic Regression
# - Finance+Text + Random Forest
# - Text-only + XGBoost
# Output: feature_importance.csv
# =========================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# =========================
# 1. Load merged modeling datasets
# =========================
TEXT_PATH = "model_text_only.csv"
COMBINED_PATH = "model_finance_text_combined.csv"

df_text = pd.read_csv(TEXT_PATH)
df_combined = pd.read_csv(COMBINED_PATH)

print("df_text shape:", df_text.shape)
print("df_combined shape:", df_combined.shape)


# =========================
# 2. Define feature columns
# =========================

# Text-only features
drop_text_cols = {
    "companyid", "companyname", "company_name", "companyid_bridge",
    "ticker", "label", "label_binary", "title",
    "pct_positive", "pct_negative", "pct_neutral",
    "mean_p_positive", "mean_p_negative", "mean_p_neutral",
    "ai_keyword_count"
}
text_feature_cols = [
    c for c in df_text.columns
    if c not in drop_text_cols and not c.startswith("PC")
]

# Combined features
drop_combined_cols = {
    "companyid", "companyname", "company_name", "companyid_bridge",
    "ticker", "label", "label_binary", "title",
    "tic", "gvkey", "datadate",
    "pct_positive", "pct_negative", "pct_neutral",
    "mean_p_positive", "mean_p_negative", "mean_p_neutral",
    "ai_keyword_count"
}
combined_feature_cols = [
    c for c in df_combined.columns
    if c not in drop_combined_cols
]

print("\nText feature count:", len(text_feature_cols))
print("Combined feature count:", len(combined_feature_cols))


# =========================
# 3. Prepare X and y
# =========================
X_text = df_text[text_feature_cols].copy()
y_text = df_text["label_binary"].copy()

X_combined = df_combined[combined_feature_cols].copy()
y_combined = df_combined["label_binary"].copy()


# =========================
# 4. Fit best text-only L1 logistic regression
# =========================
logit_l1 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="l1",
        solver="saga",
        C=1.0,
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    ))
])

logit_l1.fit(X_text, y_text)

# Extract coefficients
logit_coef = logit_l1.named_steps["clf"].coef_[0]

logit_importance = pd.DataFrame({
    "feature_set": "X_text_only",
    "model_name": "logit_l1",
    "feature": text_feature_cols,
    "raw_value": logit_coef,
    "importance": np.abs(logit_coef),
    "direction": np.where(logit_coef > 0, "positive",
                  np.where(logit_coef < 0, "negative", "zero"))
})

# Keep only non-zero coefficients for L1
logit_importance = logit_importance[logit_importance["importance"] > 0].copy()
logit_importance = logit_importance.sort_values("importance", ascending=False)

print("\nTop 15 non-zero L1 logistic coefficients:")
print(logit_importance.head(15).round(4).to_string(index=False))


# =========================
# 5. Fit best combined Random Forest
# =========================
rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42
    ))
])

rf.fit(X_combined, y_combined)

rf_importances = rf.named_steps["clf"].feature_importances_

rf_importance = pd.DataFrame({
    "feature_set": "X_fin_plus_X_text",
    "model_name": "random_forest",
    "feature": combined_feature_cols,
    "raw_value": rf_importances,
    "importance": rf_importances,
    "direction": ""
}).sort_values("importance", ascending=False)

print("\nTop 15 Random Forest importances:")
print(rf_importance.head(15).round(4).to_string(index=False))


# =========================
# 6. Fit best text-only XGBoost
# =========================
xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        eval_metric="logloss"
    ))
])

xgb.fit(X_text, y_text)

xgb_importances = xgb.named_steps["clf"].feature_importances_

xgb_importance = pd.DataFrame({
    "feature_set": "X_text_only",
    "model_name": "xgboost",
    "feature": text_feature_cols,
    "raw_value": xgb_importances,
    "importance": xgb_importances,
    "direction": ""
}).sort_values("importance", ascending=False)

print("\nTop 15 XGBoost importances:")
print(xgb_importance.head(15).round(4).to_string(index=False))


# =========================
# 7. Combine and save
# =========================
feature_importance_df = pd.concat(
    [logit_importance, rf_importance, xgb_importance],
    axis=0,
    ignore_index=True
)

# Add rank within each model
feature_importance_df["rank_within_model"] = (
    feature_importance_df
    .groupby(["feature_set", "model_name"])["importance"]
    .rank(method="first", ascending=False)
)

feature_importance_df = feature_importance_df.sort_values(
    ["feature_set", "model_name", "rank_within_model"]
)

feature_importance_df.to_csv("feature_importance.csv", index=False)

print("\nSaved file:")
print("  feature_importance.csv")

df_text shape: (154, 33)
df_combined shape: (154, 43)

Text feature count: 19
Combined feature count: 27

Top 15 non-zero L1 logistic coefficients:
feature_set model_name                                      feature  raw_value  importance direction
X_text_only   logit_l1             topic_02_ai_semiconductor_design     0.6909      0.6909  positive
X_text_only   logit_l1                                std_sentiment     0.4603      0.4603  positive
X_text_only   logit_l1                        ratio_strong_positive    -0.3781      0.3781  negative
X_text_only   logit_l1   topic_03_ai_enterprise_healthcare_services    -0.3748      0.3748  negative
X_text_only   logit_l1             topic_09_ai_cloud_infrastructure    -0.3198      0.3198  negative
X_text_only   logit_l1               topic_11_ai_saas_cybersecurity    -0.2952      0.2952  negative
X_text_only   logit_l1               topic_01_advertising_marketing    -0.2938      0.2938  negative
X_text_only   logit_l1            topic_10_c